In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score, roc_auc_score

SEED = 42
POS_WEIGHT = 3.0
NEG_PER_POS = 30
BATCH = 256

p = "../data/processed/"
tr = np.load(p + "windows_train.npz")
va = np.load(p + "windows_val.npz")
te = np.load(p + "windows_test.npz")

X_train, y_train = tr["X"], tr["y"]
X_val, y_val, s_val = va["X"], va["y"], va["serial"]
X_test, y_test, s_test = te["X"], te["y"], te["serial"]
failed_set = set(pd.read_csv(p + "fail_dates.csv")["serial_number"])

print(X_train.shape, X_val.shape, X_test.shape)
print("positives:", int(y_train.sum()), int(y_val.sum()), int(y_test.sum()))

In [ ]:
def window_metrics(y, score):
    return {"PR-AUC": round(average_precision_score(y, score), 4),
            "ROC-AUC": round(roc_auc_score(y, score), 4)}

def drive_detection(y, score, serial, thr):
    df = pd.DataFrame({"s": serial, "y": y, "sc": score})
    pos = df[df["y"] == 1].groupby("s")["sc"].max()
    healthy = df[~df["s"].isin(failed_set)].groupby("s")["sc"].max()
    return int((pos >= thr).sum()), len(pos), int((healthy >= thr).sum()), len(healthy)

def budget_table(name, sc_val, sc_test, rates=(0.01, 0.03, 0.05)):
    dfv = pd.DataFrame({"s": s_val, "sc": sc_val})
    hv = dfv[~dfv["s"].isin(failed_set)].groupby("s")["sc"].max().values
    rows = []
    for r in rates:
        thr = float(np.quantile(hv, 1 - r))
        dv, pv, fv, nv = drive_detection(y_val, sc_val, s_val, thr)
        dt, pt, ft, nt = drive_detection(y_test, sc_test, s_test, thr)
        rows.append({"method": name, "target FA": f"{r:.0%}",
                     "val detected": f"{dv}/{pv}", "val FA": f"{fv/nv:.1%}",
                     "test detected": f"{dt}/{pt}", "test FA": f"{ft/nt:.1%}"})
    return pd.DataFrame(rows)

In [ ]:
class RiskNet(nn.Module):
    def __init__(self, n_feat, hidden=32, kind="gru", dropout=0.2):
        super().__init__()
        rnn = nn.GRU if kind == "gru" else nn.LSTM
        self.rnn = rnn(n_feat, hidden, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        x = torch.clamp(x, -10.0, 10.0)
        out, _ = self.rnn(x)
        return self.head(self.drop(out[:, -1, :])).squeeze(-1)

def predict(model, X, bs=8192):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            out.append(torch.sigmoid(model(torch.from_numpy(X[i:i + bs]))).numpy())
    return np.concatenate(out)

Xtr_t = torch.from_numpy(X_train)
ytr_t = torch.from_numpy(y_train)
pos_idx = np.where(y_train == 1)[0]
neg_idx = np.where(y_train == 0)[0]

def train_model(kind, hidden=32, epochs=25, patience=5, lr=1e-3, seed=SEED):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = RiskNet(X_train.shape[-1], hidden=hidden, kind=kind)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))
    best, best_state, bad = -1.0, None, 0

    for ep in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        neg = rng.choice(neg_idx, size=len(pos_idx) * NEG_PER_POS, replace=False)
        idx = rng.permutation(np.concatenate([pos_idx, neg]))
        total = 0.0
        for i in range(0, len(idx), BATCH):
            b = torch.from_numpy(idx[i:i + BATCH])
            xb, yb = Xtr_t[b], ytr_t[b]
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item() * len(b)

        ap = average_precision_score(y_val, predict(model, X_val))
        print(f"{kind} epoch {ep:2d} | train loss {total / len(idx):.4f} | "
              f"val PR-AUC {ap:.4f} | {time.time() - t0:.0f}s")
        if ap > best:
            best = ap
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print("early stop")
                break

    model.load_state_dict(best_state)
    return model, best

In [ ]:
gru, gru_best = train_model("gru")
sc_val_gru = predict(gru, X_val)
sc_test_gru = predict(gru, X_test)
print("GRU val:", window_metrics(y_val, sc_val_gru))
print("GRU test:", window_metrics(y_test, sc_test_gru))

In [ ]:
lstm, lstm_best = train_model("lstm")
sc_val_lstm = predict(lstm, X_val)
sc_test_lstm = predict(lstm, X_test)
print("LSTM val:", window_metrics(y_val, sc_val_lstm))
print("LSTM test:", window_metrics(y_test, sc_test_lstm))

In [ ]:
tables = [budget_table("GRU", sc_val_gru, sc_test_gru),
          budget_table("LSTM", sc_val_lstm, sc_test_lstm)]
print(pd.concat(tables).to_string(index=False))